In [0]:
df = spark.sql('SELECT * FROM marathos.bronze.raw_races')

## Year of event

In [0]:
from pyspark.sql import functions as f 
from pyspark.sql.functions import col

df.select(
    f.min('Year of event').alias('earliest_year'),
    f.max('Year of event').alias('latest_year')
).show()

year_of_event_type = df.schema['Year of event'].dataType
year_of_event_nulls = df.filter(col("Year of event").isNull()).count()

print(f'Number of nulls in column "Year of event": {year_of_event_nulls}')
print(f'Data type of column "Year of event": {year_of_event_type}')

## Event dates

In [0]:
from pyspark.sql.functions import regexp_replace

df.select(
    regexp_replace("Event dates", r'\d', 'X').alias("pattern")
).groupBy("pattern").count().orderBy("count", ascending=False).show(50, truncate=False)

event_dates_nulls = df.filter(col("Event dates").isNull()).count()

print(f'Number of nulls in column "Event dates": {event_dates_nulls}')

In [0]:
from pyspark.sql.functions import concat_ws, regexp_extract, to_date
from pyspark.sql import functions as F

df = df.withColumn(
    "Event dates",
    concat_ws(".",
        regexp_extract(col("Event dates"), r"^(\d{2})", 1),
        regexp_extract(col("Event dates"), r"\.(\d{2})\.\d{4}$", 1),
        regexp_extract(col("Event dates"), r"(\d{4})$", 1),
    )
)

df = df.withColumn(
    "Event dates",
    to_date(col("Event dates"), "dd.MM.yyyy")
)

## Event name

In [0]:
event_name_nulls = df.filter(col("Event name").isNull()).count()

print(f'Number of nulls in column "Event name": {event_name_nulls}')

## Event distance/length

In [0]:
from pyspark.sql.functions import regexp_replace

df.select(
    regexp_replace(col("Event distance/length"), r'\d', 'X').alias("pattern")
).groupBy("pattern").count().orderBy("count", ascending=False).show(50, truncate=False)

In [0]:
from pyspark.sql.functions import regexp_extract_all, lit

df.select(
    regexp_extract_all(col("Event distance/length"), lit(r"[^\d./:]+"), 0).alias("non_digits")
).groupBy("non_digits").count().orderBy("count", ascending=False).show(60, truncate=False)



In [0]:
df.filter(col("Event distance/length").rlike(r"\d+:\d+h")).count()

## Event number of finishers

In [0]:
from pyspark.sql import functions as f 
from pyspark.sql.functions import col

df.select(
    f.min('Event number of finishers').alias('lowest_count'),
    f.max('Event number of finishers').alias('highest_count')
).show()

number_of_finishers_type = df.schema['Event number of finishers'].dataType
number_of_finishers_nulls = df.filter(col("Event number of finishers").isNull()).count()

print(f'Number of nulls in column "Event number of finishers": {number_of_finishers_nulls}')
print(f'The data type of column: {number_of_finishers_type}')

## Athlete club

In [0]:
from pyspark.sql.functions import col

athlete_club_type = df.schema['Athlete club'].dataType
athlete_club_nulls = df.filter(col('Athlete club').isNull()).count()

print(f'Number of nulls in column "Athlete club": {athlete_club_nulls}')
print(f'The data type of column: {athlete_club_type}')

## Athlete country

In [0]:
df.groupBy("Athlete country").count().orderBy("count", ascending=False).show(300, truncate=False)

In [0]:
from pyspark.sql.functions import upper, when, col

df = df.withColumn(
    "Athlete country",
    when(upper(col("Athlete country")).isin("XXX", "NULL"), None)
    .when(upper(col("Athlete country")) == "SVE", "SWE")
    .otherwise(upper(col("Athlete country")))
)

athlete_club_nulls = df.filter(col('Athlete country').isNull()).count()

print(f'Number of nulls in column "Athlete country": {athlete_club_nulls}')

## Athlete year of birth

In [0]:
from pyspark.sql import functions as f

df.select(
    f.min("Athlete year of birth").alias("min"),
    f.max("Athlete year of birth").alias("max")
).show()

birth_year_nulls = df.filter(df["Athlete year of birth"].isNull()).count()
print(f'Number of nulls in column "Athlete year of birth": {birth_year_nulls}')

In [0]:
from pyspark.sql.functions import col
from pyspark.sql import functions as f

df.withColumn(
    "age_at_event", 
    col("Year of event") - col("Athlete year of birth")
).select(
    f.min("age_at_event").alias("min_age"),
    f.max("age_at_event").alias("max_age")
).show()

In [0]:
df = df.withColumn(
        "Athlete year of birth", 
        when(
            (col("Year of event") - col("Athlete year of birth") < 15) |
            (col("Year of event") - col("Athlete year of birth") > 100), 
            None,
        )
        .otherwise(col("Athlete year of birth").cast("integer")),
    )

In [0]:
from pyspark.sql import functions as f
df.select(
    f.min(col("Year of event") - col("Athlete year of birth")).alias("min_age"),
    f.max(col("Year of event") - col("Athlete year of birth")).alias("max_age"),
    f.count(f.when(f.col("Athlete year of birth").isNull(), 1)).alias("null_count")
).show()

## Athlete gender

In [0]:
group_by_gender = df.groupBy("Athlete gender").count().orderBy("count", ascending=False)
group_by_gender.show()

gender_type = df.schema['Athlete gender'].dataType
print(f'The data type of column: {gender_type}')

gender_nulls = df.filter(df["Athlete gender"].isNull()).count()
print(f'Number of nulls in column "Athlete gender": {gender_nulls}')

## Athlete age category

In [0]:
age_category_group_by = df.groupBy("Athlete age category").count().orderBy("count", ascending=False)
age_category_group_by.show(50)

age_category_nulls = df.filter(df["Athlete age category"].isNull()).count()
print(f'Number of nulls in column "Athlete age category": {age_category_nulls}')

In [0]:
df = df.withColumn(
        "athlete_age_category",
        when(col("athlete_age_category") == "F35", "W35")
        .otherwise(col("athlete_age_category")),
    )

## Athlete average speed

In [0]:
from pyspark.sql.functions import col, when
from pyspark.sql import functions as F

df.select("Athlete average speed").distinct().show(50, truncate=False)

df.select(
    F.count(F.when(F.col("Athlete average speed").isNull(), 1)).alias("null_count")
).show()

In [0]:
df = df.withColumn(
    "Athlete average speed",
    when(
        col("Athlete average speed").rlike(r"^\d+:\d+"),
        None
    ).otherwise(col("Athlete average speed").cast("double"))
)

## Athlete ID

In [0]:
athlete_id_type = df.schema['Athlete ID'].dataType
print(f'The data type of column: {athlete_id_type}')

athlete_id_nulls = df.filter(df["Athlete ID"].isNull()).count()
print(f'Number of nulls in column "Athlete ID": {athlete_id_nulls}')